In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
import torch
import os
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

train_path = os.path.join('/content', 'mnist_train_small.csv')
test_path = os.path.join('/content', 'mnist_test.csv')

def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

    def remap_mask_binary(mask):
      mask_np = mask.numpy().squeeze()
      binary_mask = (mask_np != 0).astype(np.uint8)
      return torch.from_numpy(binary_mask).unsqueeze(0)

In [ ]:
import os
import glob
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import matplotlib.pyplot as plt

class SUIMDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        # Search for images and masks recursively
        self.image_paths = sorted(glob.glob(os.path.join(root_dir, '**', 'images', '*.jpg'), recursive=True) +
                                  glob.glob(os.path.join(root_dir, '**', 'images', '*.bmp'), recursive=True))
        self.mask_paths = sorted(glob.glob(os.path.join(root_dir, '**', 'masks', '*.bmp'), recursive=True))

        if len(self.image_paths) != len(self.mask_paths):
            print(f"Warning: Found {len(self.image_paths)} images and {len(self.mask_paths)} masks.")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Load Image and Mask
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L") # Load as grayscale

        target_size = (256, 256)
        image = image.resize(target_size, Image.BILINEAR)
        mask = mask.resize(target_size, Image.NEAREST)


        to_tensor = T.ToTensor()
        image = to_tensor(image)
        mask = torch.from_numpy(np.array(mask)).long()

        mask = remap_mask(mask)

        return image, mask

dataset = SUIMDataset(path)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

images, masks = next(iter(train_loader))

plt.figure(figsize=(12, 6))
for i in range(4):
    # Plot Image
    plt.subplot(2, 4, i+1)
    plt.imshow(images[i].permute(1, 2, 0)) # CHW -> HWC
    plt.axis("off")
    plt.title("Image")

    plt.subplot(2, 4, i+5)
    plt.imshow(masks[i], cmap='jet', vmin=0, vmax=7)
    plt.axis("off")
    plt.title("Ground Truth")
plt.tight_layout()
plt.show()

In [ ]:
!pip install -q segmentation_models_pytorch

In [ ]:
# TO DO
import segmentation_models_pytorch as smp

device = "cpu"
model = smp.Unet(
        encoder_name="efficientnet-b1",  # Pretrained encoder (backbone)
        encoder_weights="imagenet",  # Use ImageNet weights
        in_channels =3,
        classes = 1,
).to(device)


In [ ]:
import torch.nn as nn
import torch.optim as optim
from tqdm.notebook import tqdm

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    # Iterate over data.
    for images, masks in tqdm(dataloader, desc="Training"):
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, masks)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(dataloader.dataset)
    return epoch_loss

def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0

    with torch.no_grad():
        for images, masks in tqdm(dataloader, desc="Validation"):
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)

            running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(dataloader.dataset)
    return epoch_loss

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_epochs = 10  # You can adjust this
learning_rate = 0.001

model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

train_losses = []
val_losses = []

# Training Loop
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")

    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, data_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    print("-" * 30)

plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# TO DO